# Student Marks Prediction using Machine Learning

**Project Type:** Regression  
**Level:** College Mini Project

### Aim
To develop a Machine Learning model that predicts a student's final marks using academic factors such as study hours, attendance, previous marks, assignment score and internal test score.

### Project Pipeline
1. Problem Understanding
2. Dataset Understanding
3. Data Preprocessing
4. Exploratory Data Analysis
5. Feature and Target Selection
6. Train-Test Split
7. Model Selection
8. Model Training
9. Model Evaluation
10. Model Visualization
11. Model Improvement
12. Final Model Saving


## Part A — Problem Understanding

**Problem:** Predict a student's final marks from academic inputs.

**Input variables:**
- Study Hours
- Attendance
- Previous Exam Marks
- Assignment Score
- Internal Test Score

**Target variable:**
- Final Marks

**ML type:** Regression

**Why regression?**  
Final marks are a continuous numerical value between 0 and 100. Therefore, regression is suitable for predicting the numerical final score.


## Part B — Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data').exists():
    PROJECT_DIR = PROJECT_DIR.parent

pd.set_option("display.max_columns", None)

df = pd.read_csv(PROJECT_DIR / "data" / "student_marks.csv")
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
print("Numerical columns:")
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nBasic statistics:")
display(df.describe().round(2))


### Dataset observations

- The dataset contains student academic records.
- The target column is `Final_Marks`.
- The input columns are numerical, so no categorical encoding is required.
- Missing values and duplicate records are checked before modeling.


## Part C — Data Preprocessing

The dataset is already numerical. We will:
1. Check missing values.
2. Remove duplicate rows if present.
3. Separate features and target.
4. Use a train-test split before model training.

No categorical encoding is required because all selected features are numerical.


In [ ]:
# Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

features = [
    "Study_Hours",
    "Attendance",
    "Previous_Marks",
    "Assignment_Score",
    "Internal_Test_Score"
]
target = "Final_Marks"

X = df[features]
y = df[target]

print("Rows after duplicate handling:", len(df))
print("Missing values in selected columns:")
print(X.isnull().sum())


## Part D — Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Study_Hours"], df["Final_Marks"], alpha=0.65)
plt.xlabel("Study Hours per Day")
plt.ylabel("Final Marks")
plt.title("Study Hours vs Final Marks")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Attendance"], df["Final_Marks"], alpha=0.65)
plt.xlabel("Attendance (%)")
plt.ylabel("Final Marks")
plt.title("Attendance vs Final Marks")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df[features + [target]].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Final_Marks"], bins=15, edgecolor="black")
plt.xlabel("Final Marks")
plt.ylabel("Number of Students")
plt.title("Distribution of Final Marks")
plt.grid(axis="y", alpha=0.2)
plt.show()


### EDA Observations

1. Study hours generally show a positive relationship with final marks.
2. Higher attendance is generally associated with better final marks.
3. Previous marks, assignment scores and internal test scores also contribute to the final result.
4. The correlation heatmap helps identify which numerical features have stronger relationships with the target.


## Part E — Feature and Target Selection

`X` contains the five input features:

- Study_Hours
- Attendance
- Previous_Marks
- Assignment_Score
- Internal_Test_Score

`y` contains the target:

- Final_Marks

These features were selected because they represent measurable academic factors that can reasonably help estimate a student's final marks.


In [ ]:
X = df[features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## Part F — Train-Test Split

We use an **80% training and 20% testing** split.

The training data is used to learn patterns, while the test data is kept separate to evaluate how well the model performs on unseen student records.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## Part G — Model Selection

We will compare three regression algorithms:

1. **Linear Regression** — simple and easy to interpret.
2. **Decision Tree Regression** — can capture non-linear relationships.
3. **Random Forest Regression** — combines multiple trees and can provide stronger generalization.

Comparing multiple models helps us select the model that performs best on the test data.


## Part H — Model Training

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

print("All models trained successfully.")


## Part I — Model Evaluation

We use:
- **MAE:** Average absolute prediction error.
- **MSE:** Average squared prediction error.
- **RMSE:** Square root of MSE; easier to interpret in marks.
- **R²:** Measures how much variation in final marks is explained by the model.

For MAE, MSE and RMSE, lower values are better. For R², a value closer to 1 is better.


In [ ]:
results = []

for name, model in trained_models.items():
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df.round(3))


## Part J — Model Visualization

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Marks")
plt.ylabel("Residual")
plt.title(f"Residual Plot — {best_model_name}")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.ylabel("RMSE")
plt.title("Model Comparison using RMSE")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.2)
plt.show()


## Part K — Model Improvement

### Experiment 1
Compare a basic Decision Tree with a depth-limited Decision Tree.

### Experiment 2
Compare a single Decision Tree with Random Forest, which uses multiple trees.

The goal is to observe whether controlling tree complexity or using an ensemble improves test performance.


In [ ]:
# Experiment 1: Decision Tree depth comparison
depth_results = []

for depth in [2, 3, 5, 8, None]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    depth_results.append({
        "Max Depth": str(depth),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

depth_df = pd.DataFrame(depth_results)
display(depth_df.round(3))


In [ ]:
# Experiment 2: Random Forest tree-count comparison
forest_results = []

for n in [50, 100, 150, 250]:
    model = RandomForestRegressor(
        n_estimators=n,
        max_depth=8,
        random_state=42
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    forest_results.append({
        "Number of Trees": n,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

forest_df = pd.DataFrame(forest_results)
display(forest_df.round(3))


### Improvement conclusion

The experiments show how model complexity and the number of trees can affect prediction performance. We select the final model using test-set performance, especially RMSE and R², while avoiding unnecessarily complex settings.


## Feature Importance

For the Random Forest model, feature importance can be used to understand which input variables contributed most to predictions.


In [ ]:
rf = trained_models["Random Forest"]

importance = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

display(importance.to_frame("Importance").round(3))

plt.figure(figsize=(8, 5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.grid(axis="x", alpha=0.2)
plt.show()


## Part L — Final Interpretation

### Final model
The model with the lowest test RMSE is selected as the final model.

### Why?
Lower RMSE means the typical prediction error in marks is smaller.

### Important features
Feature importance from the Random Forest model helps identify which academic inputs contributed most strongly.

### Limitations
- The dataset is a small educational dataset.
- Real student performance depends on many factors that are not included.
- Predictions should be treated as estimates, not guaranteed marks.

### Future improvements
- Collect a larger real-world dataset.
- Include additional academic and behavioral features.
- Try hyperparameter tuning.
- Deploy the model with a more complete student management system.


In [ ]:
# Save the selected final model
model_dir = PROJECT_DIR / "model"
model_dir.mkdir(exist_ok=True)

joblib.dump(
    best_model,
    model_dir / "student_marks_model.pkl"
)

with open(model_dir / "model_info.txt", "w", encoding="utf-8") as f:
    f.write(f"Selected Model: {best_model_name}\n")
    f.write("Target: Final_Marks\n")
    f.write("Features: " + ", ".join(features) + "\n")

print("Final model:", best_model_name)
print("Saved model to:", model_dir / "student_marks_model.pkl")


## Final Test Prediction

This cell demonstrates how a new student's values can be passed to the saved model.


In [ ]:
sample_student = pd.DataFrame([{
    "Study_Hours": 5.0,
    "Attendance": 85.0,
    "Previous_Marks": 72.0,
    "Assignment_Score": 80.0,
    "Internal_Test_Score": 75.0
}])

sample_prediction = best_model.predict(sample_student)[0]
print(f"Predicted Final Marks: {sample_prediction:.2f} / 100")
